# SaaS Product Engagement & Conversion Analysis
## Part 1: Data Cleaning & Exploratory Data Analysis

**Business Question:** Which user behaviors during onboarding predict conversion and long-term retention, and where are users dropping off in the activation funnel?

**Datasets:**
- Primary: SaaS Subscription & Churn Analytics (Kaggle/rivalytics)
- Secondary: User Funnels Dataset (Kaggle/amirmotefaker)


---
## Step 1: Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Plot styling
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
PALETTE = ['#2E75B6', '#27AE60', '#E74C3C', '#F39C12', '#8E44AD', '#1ABC9C']

print('Libraries loaded successfully.')

---
## Step 2: Load & Inspect Raw Data

**Action item:** Download both CSVs from Kaggle and place them in `../data/raw/`.  
Update the filenames below to match what you downloaded.

In [ ]:
# ============================================================
# UPDATE THESE FILENAMES to match your downloaded CSVs
# ============================================================
SUBSCRIPTION_FILE = '../data/raw/saas_subscription_data.csv'  # <-- update
FUNNEL_FILE       = '../data/raw/user_funnels_data.csv'       # <-- update

df_sub = pd.read_csv(SUBSCRIPTION_FILE)
print(f'Subscription data: {df_sub.shape[0]:,} rows x {df_sub.shape[1]} columns')
print(f'Columns: {list(df_sub.columns)}\n')

df_funnel = pd.read_csv(FUNNEL_FILE)
print(f'Funnel data: {df_funnel.shape[0]:,} rows x {df_funnel.shape[1]} columns')
print(f'Columns: {list(df_funnel.columns)}')

In [ ]:
# First look at both datasets
print('=' * 60)
print('SUBSCRIPTION DATA')
print('=' * 60)
display(df_sub.head())

print('\n' + '=' * 60)
print('FUNNEL DATA')
print('=' * 60)
display(df_funnel.head())

---
## Step 3: Data Profiling

Understand data types, nulls, distributions, and quality issues before cleaning.

In [ ]:
print('=' * 60)
print('SUBSCRIPTION DATA - Profile')
print('=' * 60)
print(df_sub.dtypes)
print('\n--- Null Counts ---')
print(df_sub.isnull().sum())
print(f'\nDuplicate rows: {df_sub.duplicated().sum()}')
print(f'\n--- Categorical Value Counts ---')
for col in df_sub.select_dtypes(include="object").columns:
    print(f'\n{col}:')
    print(df_sub[col].value_counts().head(10))

In [ ]:
print('=' * 60)
print('SUBSCRIPTION DATA - Descriptive Statistics')
print('=' * 60)
display(df_sub.describe(include='all').T)

In [ ]:
print('=' * 60)
print('FUNNEL DATA - Profile')
print('=' * 60)
print(df_funnel.dtypes)
print('\n--- Null Counts ---')
print(df_funnel.isnull().sum())
print(f'\nDuplicate rows: {df_funnel.duplicated().sum()}')

In [ ]:
# Auto-detect and parse date columns
for df_name, df in [('Subscription', df_sub), ('Funnel', df_funnel)]:
    date_keywords = ['date', 'time', 'created', 'signup', 'start', 'end', 'cancel']
    date_cols = [c for c in df.columns if any(kw in c.lower() for kw in date_keywords)]
    if date_cols:
        print(f'\n{df_name} - Potential date columns: {date_cols}')
        for col in date_cols:
            try:
                df[col] = pd.to_datetime(df[col])
                print(f'  Parsed "{col}" -> range: {df[col].min()} to {df[col].max()}')
            except Exception as e:
                print(f'  Could not parse "{col}": {e}')

---
## Step 4: Data Cleaning

- Standardize column names (snake_case)
- Handle nulls
- Remove duplicates
- Fix data types
- Validate categorical values

In [ ]:
# Standardize column names
import re
def clean_columns(df):
    df.columns = [re.sub(r'[^a-z0-9]+', '_', c.strip().lower()).strip('_') for c in df.columns]
    return df

df_sub = clean_columns(df_sub)
df_funnel = clean_columns(df_funnel)

print('Subscription columns:', list(df_sub.columns))
print('Funnel columns:      ', list(df_funnel.columns))

In [ ]:
# Handle nulls
print('--- Handling Subscription nulls ---')
for col in df_sub.select_dtypes(include='number').columns:
    n = df_sub[col].isnull().sum()
    if n > 0:
        fill = df_sub[col].median()
        df_sub[col].fillna(fill, inplace=True)
        print(f'  Filled {n} nulls in "{col}" with median ({fill:.2f})')

for col in df_sub.select_dtypes(include='object').columns:
    n = df_sub[col].isnull().sum()
    if n > 0:
        df_sub[col].fillna('Unknown', inplace=True)
        print(f'  Filled {n} nulls in "{col}" with "Unknown"')

print(f'\nRemaining nulls: {df_sub.isnull().sum().sum()}')

In [ ]:
# Remove duplicates
for name, df in [('Subscription', df_sub), ('Funnel', df_funnel)]:
    before = len(df)
    df.drop_duplicates(inplace=True)
    print(f'{name}: removed {before - len(df)} duplicates ({len(df):,} remaining)')

---
## Step 5: Feature Engineering

Create the derived columns that power your funnel, cohort, and segmentation analysis.  
**Uncomment and update column names** to match your actual data.

In [ ]:
# ============================================================
# FEATURE ENGINEERING - Uncomment & update column references
# ============================================================

# --- Signup Cohort (monthly) ---
# df_sub['signup_cohort'] = df_sub['signup_date'].dt.to_period('M')

# --- Tenure in Days ---
# df_sub['tenure_days'] = (df_sub['end_date'] - df_sub['start_date']).dt.days

# --- Revenue per Day ---
# df_sub['revenue_per_day'] = df_sub['monthly_revenue'] / 30

# --- Binary Churn Flag ---
# df_sub['is_churned'] = (df_sub['status'] == 'Churned').astype(int)

# --- Engagement Score (composite) ---
# df_sub['engagement_score'] = (
#     df_sub['logins_30d'] * 0.3 +
#     df_sub['features_used'] * 0.4 +
#     df_sub['support_tickets'].apply(lambda x: max(0, 5-x)) * 0.3
# )

print('Available columns for feature engineering:')
print(list(df_sub.columns))

---
## Step 6: Exploratory Data Analysis

Key questions:
1. What is the overall churn rate?
2. How does churn vary by plan tier?
3. What is the revenue distribution?
4. Are there seasonal signup patterns?
5. Which features correlate with churn?

In [ ]:
# ============================================================
# EDA PLOT 1: Churn Rate Overview
# Update column names below
# ============================================================

# churn_col = 'is_churned'       # update
# plan_col  = 'subscription_type' # update
# 
# churn_rate = df_sub[churn_col].mean()
# 
# fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# 
# df_sub[churn_col].value_counts().plot.pie(
#     ax=axes[0], autopct='%1.1f%%', colors=[PALETTE[0], PALETTE[2]],
#     labels=['Active', 'Churned'], startangle=90)
# axes[0].set_title(f'Overall Churn Rate: {churn_rate:.1%}')
# axes[0].set_ylabel('')
# 
# df_sub.groupby(plan_col)[churn_col].mean().sort_values().plot.barh(
#     ax=axes[1], color=PALETTE[0])
# axes[1].set_title('Churn Rate by Plan Tier')
# axes[1].set_xlabel('Churn Rate')
# 
# plt.tight_layout()
# plt.savefig('../data/cleaned/eda_churn_overview.png', dpi=150, bbox_inches='tight')
# plt.show()

print('Uncomment after updating column names.')

In [ ]:
# ============================================================
# EDA PLOT 2: Revenue Distribution by Plan
# ============================================================

# revenue_col = 'monthly_revenue'  # update
# plan_col = 'subscription_type'   # update
# 
# fig, ax = plt.subplots(figsize=(12, 5))
# for i, plan in enumerate(df_sub[plan_col].unique()):
#     subset = df_sub[df_sub[plan_col] == plan][revenue_col]
#     ax.hist(subset, bins=30, alpha=0.6, label=plan, color=PALETTE[i])
# 
# ax.set_title('Monthly Revenue Distribution by Plan')
# ax.set_xlabel('Monthly Revenue ($)')
# ax.set_ylabel('Customer Count')
# ax.legend()
# plt.tight_layout()
# plt.savefig('../data/cleaned/eda_revenue_dist.png', dpi=150, bbox_inches='tight')
# plt.show()

print('Uncomment after updating column names.')

In [ ]:
# ============================================================
# EDA PLOT 3: Signup Cohort Trend
# ============================================================

# date_col = 'signup_date'  # update
# 
# monthly = df_sub.set_index(date_col).resample('M').size()
# fig, ax = plt.subplots(figsize=(12, 5))
# monthly.plot(kind='bar', color=PALETTE[0], ax=ax)
# ax.set_title('Monthly Signup Volume')
# ax.set_xlabel('Month')
# ax.set_ylabel('New Signups')
# ax.tick_params(axis='x', rotation=45)
# plt.tight_layout()
# plt.savefig('../data/cleaned/eda_signup_trend.png', dpi=150, bbox_inches='tight')
# plt.show()

print('Uncomment after updating column names.')

In [ ]:
# ============================================================
# EDA PLOT 4: Correlation Heatmap
# ============================================================

numeric_df = df_sub.select_dtypes(include='number')
if len(numeric_df.columns) > 2:
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(numeric_df.corr(), annot=True, cmap='RdBu_r', center=0,
                fmt='.2f', ax=ax, square=True)
    ax.set_title('Feature Correlation Matrix')
    plt.tight_layout()
    plt.savefig('../data/cleaned/eda_correlation.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Not enough numeric columns yet.')

---
## Step 7: Export Cleaned Data

Save cleaned CSVs for:
- Loading into MySQL via DBeaver (SQL analysis phase)
- Connecting to Tableau (dashboard phase)

In [ ]:
df_sub.to_csv('../data/cleaned/subscriptions_cleaned.csv', index=False)
df_funnel.to_csv('../data/cleaned/funnels_cleaned.csv', index=False)

print(f'Exported subscriptions_cleaned.csv ({len(df_sub):,} rows)')
print(f'Exported funnels_cleaned.csv ({len(df_funnel):,} rows)')
print()
print('NEXT STEPS:')
print('  1. Import cleaned CSVs into MySQL via DBeaver')
print('  2. Run SQL queries from the sql/ folder')
print('  3. Connect Tableau to MySQL or cleaned CSVs')
print('  4. Build dashboard views')